# Mapping VSIC VN to MCC on Google Colab

Notebook này hướng dẫn cách chạy pipeline mapping mã ngành VSIC Việt Nam sang mã MCC (Merchant Category Code) sử dụng GPU trên Google Colab và Ollama.

## 1. Kết nối Google Drive
Kết quả mapping và checkpoint sẽ được lưu trực tiếp lên Google Drive để tránh mất dữ liệu khi runtime bị reset.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Cài đặt Ollama
Chúng ta sẽ cài đặt Ollama để chạy các mô hình LLM local trên GPU của Colab.

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

## 3. Khởi động Ollama Server
Chạy Ollama server trong background.

In [ ]:
import os
import threading
import time

def run_ollama():
    os.system("ollama serve")

threading.Thread(target=run_ollama, daemon=True).start()
time.sleep(5) # Đợi server khởi động

## 4. Tải Mô hình (Models)
Tải mô hình Qwen 3.5 9B cho việc re-ranking và BGE-M3 cho embedding.

In [ ]:
!ollama pull qwen3.5:9b
!ollama pull bge-m3

## 5. Cài đặt Dependencies dự án
Clone code (nếu cần) hoặc cài đặt các thư viện Python cần thiết.

In [ ]:
# Giả sử bạn đã upload code lên Drive hoặc clone vào /content
%cd /content/drive/MyDrive/projects/mcc-lens
!pip install -r requirements.txt

## 6. Chạy Pipeline Mapping
Sử dụng flag `--gdrive-output-dir` để lưu kết quả vào thư mục trên Drive.

In [ ]:
!python3 main.py map-vsic-mcc \
  --vsic-input output/vsic-vn.json \
  --mcc-input output/mcc-visa.json \
  --gdrive-output-dir /content/drive/MyDrive/projects/mcc-lens \
  --llm-model qwen3.5:9b \
  --resume